In [3]:
import polars as pl
import matplotlib.pyplot as plt
import seaborn as sns
from wordcloud import WordCloud
from collections import Counter, defaultdict
from gensim.parsing.preprocessing import STOPWORDS
from tqdm import tqdm
from gensim.corpora import Dictionary
from gensim.models import LdaMulticore
from gensim.models.coherencemodel import CoherenceModel
from gensim.models.phrases import Phraser
import pyLDAvis
import pyLDAvis.gensim_models
import os
from bertopic import BERTopic
import math
import concurrent.futures

In [3]:
csv_reviews = '../data/csv/yelp_academic_dataset_review.csv'
base_models_dir = '../data/models'

yelp_stopwords = [
    'food', 'good', 'place', 'service', 'restaurant', 'great', 'time', 
    'go', 'back', 'really', 'just', 'like', 'get', 'one', 'would', 
    'ive', 'even', 'also', 'always', 'got', 'came', 'went', 'us', 'im', 'much'
]
stopwords_list = list(STOPWORDS) + [''] + yelp_stopwords

CATEGORIES = {
    'fast_food':   'Fast Food',
    'steakhouses': 'Steakhouses',
}

def calculate_topic_diversity(topic_words_list):
    if not topic_words_list:
        return 0.0
    all_words = [word for topic in topic_words_list for word in topic]
    unique_words = set(all_words)
    return len(unique_words) / len(all_words)

def evaluate_coherence_and_diversity(input_csv, models_dir, categories_dict, sample_size=100000):
    
    sample_df = (
        pl.scan_csv(input_csv)
        .select('text')
        .collect()
        .sample(n=sample_size, seed=42)
        .with_columns(
            pl.col('text')
            .str.replace_all(r'[^a-zA-Z\s]', '')
            .str.to_lowercase()
            .str.split(' ')
            .list.set_difference(stopwords_list)
            .alias('tokens')
        )
    )
    reference_texts = sample_df['tokens'].to_list()
    del sample_df
    
    global_dict = Dictionary(reference_texts)
    top_n_words = 10
    results = []

    for category_key in categories_dict.keys():
        print(f'Evaluando categoría: {category_key} ---')
        
        lda_path = os.path.join(models_dir, category_key, 'lda', 'lda_model.gensim')
        dict_path = os.path.join(models_dir, category_key, 'lda', 'dictionary.dict')
        bigram_path = os.path.join(models_dir, category_key, 'lda', 'bigram_model.pkl')
        bertopic_path = os.path.join(models_dir, category_key, 'bertopic', 'bertopic_model.pkl') 


        if os.path.exists(lda_path) and os.path.exists(dict_path) and os.path.exists(bigram_path):
            try:
                lda = LdaMulticore.load(lda_path)
                dictionary = Dictionary.load(dict_path)
                bigram_model = Phraser.load(bigram_path)
                
                lda_reference_texts = [bigram_model[doc] for doc in reference_texts]
                
                lda_topics = []
                for topic_id in range(lda.num_topics):
                    words = [dictionary[word_id] for word_id, _ in lda.get_topic_terms(topic_id, topn=top_n_words)]
                    lda_topics.append(words)

                lda_diversity = calculate_topic_diversity(lda_topics)
                
                cm_lda = CoherenceModel(
                    topics=lda_topics, 
                    texts=lda_reference_texts,
                    dictionary=dictionary, 
                    coherence='c_v'
                )
                lda_coherence = cm_lda.get_coherence()
                
                results.append({
                    'Categoría': category_key,
                    'Modelo': 'LDA',
                    'Coherencia (C_v)': round(lda_coherence, 4),
                    'Diversidad': round(lda_diversity, 4)
                })
                print(f'LDA evaluado.')
            except Exception as e:
                print(f'Error evaluando LDA para {category_key}: {e}')
        else:
            print(f'Faltan archivos de LDA para {category_key}.')

        if os.path.exists(bertopic_path):
            try:
                bertopic_model = BERTopic.load(bertopic_path)
                
                bertopic_topics = []
                for topic_id in bertopic_model.get_topic_info()['Topic']:
                    if topic_id != -1: 
                        rep = bertopic_model.get_topic(topic_id)
                        if rep:
                            words = [word for word, _ in rep[:top_n_words]]
                            bertopic_topics.append(words)

                bertopic_diversity = calculate_topic_diversity(bertopic_topics)

                valid_bertopic_topics = []
                for topic in bertopic_topics:
                    valid_words = []
                    for phrase in topic:
                        sub_words = phrase.split(' ')
                        if all(w in global_dict.token2id for w in sub_words):
                            valid_words.append(phrase)
                            
                    if len(valid_words) >= 2:
                        valid_bertopic_topics.append(valid_words)

                results.append({
                    'Categoría': category_key,
                    'Modelo': 'BERTopic',
                    'Coherencia (C_v)': 0.0,
                    'Diversidad': round(bertopic_diversity, 4)
                })
                print(f'BERTopic evaluado.')
            except Exception as e:
                print(f'Error evaluando BERTopic para {category_key}: {e}')
        else:
            print(f'Faltan archivos de BERTopic para {category_key}.')

    df_results = pl.DataFrame(results)
    print(df_results)

final_metrics_df = evaluate_coherence_and_diversity(csv_reviews, base_models_dir, CATEGORIES)

Pre-procesando textos de referencia globales (esto tomará un momento, pero solo se hace una vez)...

--- Evaluando categoría: fast_food ---


/usr/lib/python3.12/multiprocessing/popen_fork.py:66: DeprecationWarning: This process (pid=32650) is multi-threaded, use of fork() may lead to deadlocks in the child.
  self.pid = os.fork()
/usr/lib/python3.12/multiprocessing/popen_fork.py:66: DeprecationWarning: This process (pid=32650) is multi-threaded, use of fork() may lead to deadlocks in the child.
  self.pid = os.fork()
/usr/lib/python3.12/multiprocessing/popen_fork.py:66: DeprecationWarning: This process (pid=32650) is multi-threaded, use of fork() may lead to deadlocks in the child.
  self.pid = os.fork()
/usr/lib/python3.12/multiprocessing/popen_fork.py:66: DeprecationWarning: This process (pid=32650) is multi-threaded, use of fork() may lead to deadlocks in the child.
  self.pid = os.fork()
/usr/lib/python3.12/multiprocessing/popen_fork.py:66: DeprecationWarning: This process (pid=32650) is multi-threaded, use of fork() may lead to deadlocks in the child.
  self.pid = os.fork()
/usr/lib/python3.12/multiprocessing/popen_for

[✓] LDA evaluado.


Loading weights: 100%|█████████████████████████████████████████████| 103/103 [00:00<00:00, 992.72it/s, Materializing param=pooler.dense.weight]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
/usr/lib/python3.12/multiprocessing/popen_fork.py:66: DeprecationWarning: This process (pid=32650) is multi-threaded, use of fork() may lead to deadlocks in the child.
  self.pid = os.fork()
/usr/lib/python3.12/multiprocessing/popen_fork.py:66: DeprecationWarning: This process (pid=32650) is multi-threaded, use of fork() may lead to deadlocks in the child.
  self.pid = os.fork()
/usr/lib/python3.12/multiprocessing/popen_fork.py:66: DeprecationWarning: This process (pid=32650) is multi-threaded, use of fork() may lead to deadlocks in 

[✓] BERTopic evaluado.

--- Evaluando categoría: steakhouses ---


/usr/lib/python3.12/multiprocessing/popen_fork.py:66: DeprecationWarning: This process (pid=32650) is multi-threaded, use of fork() may lead to deadlocks in the child.
  self.pid = os.fork()
/usr/lib/python3.12/multiprocessing/popen_fork.py:66: DeprecationWarning: This process (pid=32650) is multi-threaded, use of fork() may lead to deadlocks in the child.
  self.pid = os.fork()
/usr/lib/python3.12/multiprocessing/popen_fork.py:66: DeprecationWarning: This process (pid=32650) is multi-threaded, use of fork() may lead to deadlocks in the child.
  self.pid = os.fork()
/usr/lib/python3.12/multiprocessing/popen_fork.py:66: DeprecationWarning: This process (pid=32650) is multi-threaded, use of fork() may lead to deadlocks in the child.
  self.pid = os.fork()
/usr/lib/python3.12/multiprocessing/popen_fork.py:66: DeprecationWarning: This process (pid=32650) is multi-threaded, use of fork() may lead to deadlocks in the child.
  self.pid = os.fork()
/usr/lib/python3.12/multiprocessing/popen_for

[✓] LDA evaluado.


Loading weights: 100%|████████████████████████████████████████████| 103/103 [00:00<00:00, 1061.76it/s, Materializing param=pooler.dense.weight]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
/usr/lib/python3.12/multiprocessing/popen_fork.py:66: DeprecationWarning: This process (pid=32650) is multi-threaded, use of fork() may lead to deadlocks in the child.
  self.pid = os.fork()
/usr/lib/python3.12/multiprocessing/popen_fork.py:66: DeprecationWarning: This process (pid=32650) is multi-threaded, use of fork() may lead to deadlocks in the child.
  self.pid = os.fork()
/usr/lib/python3.12/multiprocessing/popen_fork.py:66: DeprecationWarning: This process (pid=32650) is multi-threaded, use of fork() may lead to deadlocks in 

[✓] BERTopic evaluado.

=== RESULTADOS FINALES ===
shape: (4, 4)
┌─────────────┬──────────┬──────────────────┬────────────┐
│ Categoría   ┆ Modelo   ┆ Coherencia (C_v) ┆ Diversidad │
│ ---         ┆ ---      ┆ ---              ┆ ---        │
│ str         ┆ str      ┆ f64              ┆ f64        │
╞═════════════╪══════════╪══════════════════╪════════════╡
│ fast_food   ┆ LDA      ┆ 0.4739           ┆ 0.4        │
│ fast_food   ┆ BERTopic ┆ 0.4024           ┆ 0.562      │
│ steakhouses ┆ LDA      ┆ 0.5158           ┆ 0.44       │
│ steakhouses ┆ BERTopic ┆ 0.3838           ┆ 0.5983     │
└─────────────┴──────────┴──────────────────┴────────────┘


In [5]:
csv_reviews = '../data/csv/yelp_academic_dataset_review.csv'
base_models_dir = '../data/models'
base_vis_dir = '../results/topic_modeling/visualizations'
stopwords_list = list(STOPWORDS) + ['']

yelp_stopwords = [
    'food', 'good', 'place', 'service', 'restaurant', 'great', 'time', 
    'go', 'back', 'really', 'just', 'like', 'get', 'one', 'would', 
    'ive', 'even', 'also', 'always', 'got', 'came', 'went', 'us', 'im', 'much'
]
stopwords_list = list(STOPWORDS) + [''] + yelp_stopwords

CATEGORIES = {
    'fast_food':   'Fast Food',
    'steakhouses': 'Steakhouses',
}

def generate_visualizations(input_csv, models_dir, vis_dir, categories_dict, sample_size=50000):
    
    sample_df = (
        pl.scan_csv(input_csv)
        .select('text')
        .collect()
        .sample(n=sample_size, seed=42)
        .with_columns(
            pl.col('text')
            .str.replace_all(r'[^a-zA-Z\s]', '')
            .str.to_lowercase()
            .str.split(' ')
            .list.set_difference(stopwords_list)
            .alias('tokens')
        )
    )
    
    tokenized_texts = sample_df['tokens'].to_list()
    del sample_df  

    for category_key in categories_dict.keys():
        print(f'Visualizaciones para: {category_key}')
        
        cat_model_dir = os.path.join(models_dir, category_key)
        lda_path = os.path.join(cat_model_dir, 'lda', 'lda_model.gensim')
        dict_path = os.path.join(cat_model_dir, 'lda', 'dictionary.dict')
        bigram_path = os.path.join(cat_model_dir, 'lda', 'bigram_model.pkl') # <-- NEW
        
        bertopic_path = os.path.join(cat_model_dir, 'bertopic', 'bertopic_model.pkl') 
        
        cat_vis_dir = os.path.join(vis_dir, category_key)
        os.makedirs(cat_vis_dir, exist_ok=True)
        
        lda_html_output = os.path.join(cat_vis_dir, 'lda_intertopic_map.html')
        intertopic_html_output = os.path.join(cat_vis_dir, 'bertopic_intertopic_map.html')
        words_html_output = os.path.join(cat_vis_dir, 'bertopic_topic_words.html')

        if os.path.exists(lda_path) and os.path.exists(dict_path) and os.path.exists(bigram_path):
            try:
                lda = LdaMulticore.load(lda_path)
                dictionary = Dictionary.load(dict_path)
                bigram_model = Phraser.load(bigram_path)
                
                cat_tokenized_texts = [bigram_model[doc] for doc in tokenized_texts]
                
                bow_corpus = [dictionary.doc2bow(text) for text in cat_tokenized_texts]
                
                vis_data_lda = pyLDAvis.gensim_models.prepare(
                    lda, 
                    bow_corpus, 
                    dictionary, 
                    mds='pcoa',
                    R=10
                )
                pyLDAvis.save_html(vis_data_lda, lda_html_output)
                print(f'LDA visualización guardada en: {lda_html_output}')
            except Exception as e:
                print(f'Error generando LDA vis para {category_key}: {e}')
        else:
            print(f'Faltan archivos de LDA para {category_key}, saltando...')

        if os.path.exists(bertopic_path):
            try:
                bertopic_model = BERTopic.load(bertopic_path)
                
                fig_intertopic = bertopic_model.visualize_topics(top_n_topics=15)
                fig_words = bertopic_model.visualize_barchart(top_n_topics=15, n_words=5)
                    
                fig_intertopic.write_html(intertopic_html_output)
                fig_words.write_html(words_html_output)
                
                print(f'BERTopic visualizaciones guardadas en: {cat_vis_dir}')
            except Exception as e:
                print(f'Error generando BERTopic vis para {category_key}: {e}')
        else:
            print(f'Faltan archivos de BERTopic para {category_key}, saltando...')

generate_visualizations(csv_reviews, base_models_dir, base_vis_dir, CATEGORIES)

Visualizaciones para: fast_food
LDA visualización guardada en: ../results/topic_modeling/visualizations/fast_food/lda_intertopic_map.html
BERTopic visualizaciones guardadas en: ../results/topic_modeling/visualizations/fast_food
Visualizaciones para: steakhouses
LDA visualización guardada en: ../results/topic_modeling/visualizations/steakhouses/lda_intertopic_map.html
BERTopic visualizaciones guardadas en: ../results/topic_modeling/visualizations/steakhouses
